# 09 — Combined model: physics + Gaussian Process (ensembles)

Notebook 08 found: **GP beats physics on water** (3.24 vs 3.96 %, p = 0.002) but loses on oil and gas.
Here we test ways to combine them. Blend rule per target (`vfm.make_blend_method`):
```
rate = (1 − w) · physics + w · GP          w = 0 → physics only, 1 → GP only, 0.5 → average
```
Averaging helps when the two models make *different* errors: part of each error cancels.

| Experiment | oil | water | gas |
|---|---|---|---|
| E1 | physics | GP | physics |
| E2 | physics | ½ physics + ½ GP | physics |
| E3 | ½ + ½ | ½ + ½ | ½ + ½ |
| E4 | physics | GP that also gets the physics water prediction as input (OOF) | physics |
| E5 | physics | ½ physics + ½ (GP + physics input) | physics |

⏱ ~3 minutes.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..', 'src').resolve()))
import load
import validation as V
import vfm

tt = pd.read_parquet(load.PROCESSED / 'test_features.parquet')
PF = vfm.PHYSICS_FEATURES

gp = vfm.make_ml_method(PF, vfm.gaussian_process, use_well_id=True, well_encoding='onehot')
gp_with_physics = vfm.make_physics_feature_method(PF, vfm.gaussian_process, out_of_fold=True,
                                                  well_encoding='onehot')

def weights(oil, water, gas):
    return {'oil_rate_stbd': oil, 'water_rate_stbd': water, 'gas_rate_mscfd': gas}

experiments = {
    'B physics (reference)': vfm.physics,
    'E1 physics oil/gas + GP water': vfm.make_blend_method(vfm.physics, gp, weights(0, 1, 0)),
    'E2 physics oil/gas + 50/50 water': vfm.make_blend_method(vfm.physics, gp, weights(0, 0.5, 0)),
    'E3 50/50 physics+GP all targets': vfm.make_blend_method(vfm.physics, gp, weights(0.5, 0.5, 0.5)),
    'E4 physics oil/gas + GP(+phys input) water': vfm.make_blend_method(vfm.physics, gp_with_physics, weights(0, 1, 0)),
    'E5 physics oil/gas + 50/50 GP(+phys input) water': vfm.make_blend_method(vfm.physics, gp_with_physics, weights(0, 0.5, 0)),
}

results, predictions = [], {}
for name, method in experiments.items():
    predictions[name], _ = V.run_experiment(name, tt, method, results)
    print(f'{name:50s} mean MAPE {results[-1]["mean_MAPE"]:.2f}', flush=True)

V.save_results(results, load.ROOT / 'outputs' / 'ablation_results.csv')
pd.DataFrame(results)[['experiment', 'oil_MAPE', 'water_MAPE', 'gas_MAPE', 'mean_MAPE']].round(2)

## Significance of each combination vs physics (water is the target that changes)

In [ ]:
base = predictions['B physics (reference)']
rows = []
for name in list(experiments)[1:]:
    test = V.paired_test(tt, base, predictions[name])
    for target in V.TARGETS:
        rows.append({'experiment': name, 'target': target, **test.loc[target].to_dict()})
sig = pd.DataFrame(rows)
sig[sig.mean_diff_pts != 0].round(3)

## Is the water gain broad (most wells) or driven by one or two wells?

In [ ]:
per_well = pd.DataFrame({
    'physics': V.score_per_well(tt, base)['water_rate_stbd'],
    'E1 GP': V.score_per_well(tt, predictions['E1 physics oil/gas + GP water'])['water_rate_stbd'],
    'E2 50/50': V.score_per_well(tt, predictions['E2 physics oil/gas + 50/50 water'])['water_rate_stbd'],
})
per_well['E2 better?'] = per_well['E2 50/50'] < per_well['physics']
per_well.round(2)

## Water blend weight sweep (w = share of GP)
⚠ Picking the best w from this same curve would be slightly optimistic (we'd be tuning on the validation
results). It's shown to see how *sensitive* the result is: a flat curve means any w in that range is safe.

In [ ]:
gp_pred = predictions['E1 physics oil/gas + GP water']           # its water column = pure GP
true = tt.water_rate_stbd
ws = np.linspace(0, 1, 11)
mape = [(((1 - w) * base.water_rate_stbd + w * gp_pred.water_rate_stbd - true).abs() / true).mean() * 100
        for w in ws]
plt.figure(figsize=(6, 3.5))
plt.plot(ws, mape, 'o-')
plt.xlabel('w (share of GP in water prediction)')
plt.ylabel('water MAPE %')
plt.title('Water error vs blend weight')
plt.grid(alpha=0.3)
plt.show()
pd.Series(mape, index=ws.round(1), name='water MAPE %').round(2)

**Conclusions**
- Every combination that uses GP for water beats physics on water, and the gain is **significant**:
  E1 (GP water) 3.96 -> 3.24 %, p = 0.002; **E2 (50/50 water) 3.96 -> 3.36 %, better on 58/85 tests, p < 0.001**.
- **Broad, not one lucky well:** 50/50 water is better than physics in **9 of 10 wells** (only GL-08 slightly worse).
- Giving GP the physics water prediction as input (E4/E5) adds nothing.
- Blending oil and gas too (E3) makes them slightly worse -> keep **physics for oil and gas**.
- Weight sweep: water error is flat for w = 0.6 ... 1.0 (3.23 - 3.31 %) -> result is not sensitive to the exact weight.
- **Chosen VFM model: physics for oil & gas, 50/50 physics + GP for water (E2), mean MAPE 3.84 %.**
  Why 50/50 and not pure GP (3.24): w = 0.5 was fixed in advance (not tuned on these results), it is the
  most consistent winner (58/85 tests, 9/10 wells), and it keeps the physics anchor if GP misbehaves on
  unusual days (GP far from its training points falls back to an average).
